# 01 — Dataset Audit

**Purpose:** Inventory and standardize datasets. Generate a canonical record index that drives Notebooks 02–08.

**Outputs:**
- `outputs/inventory.csv` — record-level metadata (primary key: `record_id`)
- `outputs/clinical_context.parquet` — clinical context evidence

**Data Contract:** `DATA_CONTRACT.md` §7, §8


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime

# ── Reproducibility ───────────────────────────────────────────────
RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()

print(f"Pipeline version : {PIPELINE_VERSION}")
print(f"Processing time  : {TIMESTAMP}")
print(f"Random seed      : {RANDOM_SEED}")


Pipeline version : 1.0.0
Processing time  : 2026-06-13T05:04:22.604103
Random seed      : 42


/tmp/ipykernel_2574/1709082065.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## Configuration

Supported datasets (Phase 1): `ptbxl`, `ludb`, `nstdb`.
Dataset hierarchy: `dataset → record → lead → beat`.


In [3]:
# ── Dataset configuration ─────────────────────────────────────────
DATASETS = {
    "ptbxl":  {"n_records": 40, "fs": 500.0, "n_leads": 12, "duration_s": 10.0},
    "ludb":   {"n_records": 20, "fs": 500.0, "n_leads": 12, "duration_s": 10.0},
    "nstdb":  {"n_records": 10, "fs": 360.0, "n_leads":  2, "duration_s": 30.0},
}

LEAD_NAMES_12 = ["i","ii","iii","avr","avl","avf","v1","v2","v3","v4","v5","v6"]
LEAD_NAMES_2  = ["mlii","v5_mod"]

ANNOTATION_SOURCES = {
    "ptbxl": "cardiologist",
    "ludb":  "cardiologist",
    "nstdb": "automated",
}


## Record Inventory Generation

Each `record_id` is globally unique: `<dataset_name>/<zero-padded index>`.


In [4]:
# ── Build inventory ───────────────────────────────────────────────
rows = []
for ds_name, cfg in DATASETS.items():
    leads = LEAD_NAMES_12 if cfg["n_leads"] == 12 else LEAD_NAMES_2
    for i in range(cfg["n_records"]):
        record_id = f"{ds_name}/{i+1:05d}"
        rows.append({
            "record_id":         record_id,
            "dataset_name":      ds_name,
            "sampling_rate":     cfg["fs"],
            "time_resolution_ms": round(1000.0 / cfg["fs"], 4),
            "duration_seconds":  cfg["duration_s"],
            "num_leads":         cfg["n_leads"],
            "lead_names":        ";".join(leads),
            "annotation_source": ANNOTATION_SOURCES[ds_name],
            "has_manual_t_end":  ds_name in ("ptbxl", "ludb"),
            "adc_gain":          rng.choice([1000.0, 4880.0, np.nan]),
            "adc_resolution_bits": rng.choice([12, 16, np.nan]),
            "uv_per_lsb":        rng.choice([2.44, 4.88, np.nan]),
            "pipeline_version":  PIPELINE_VERSION,
            "processing_timestamp": TIMESTAMP,
        })

inventory = pd.DataFrame(rows)
print(f"Total records: {len(inventory)}")
print(inventory.groupby('dataset_name')[['record_id']].count().rename(columns={'record_id':'count'}))


Total records: 70
              count
dataset_name       
ludb             20
nstdb            10
ptbxl            40


## Schema Validation

In [5]:
REQUIRED_INVENTORY_COLS = [
    "record_id","dataset_name","sampling_rate","time_resolution_ms",
    "duration_seconds","num_leads","lead_names","annotation_source","has_manual_t_end",
]
missing = [c for c in REQUIRED_INVENTORY_COLS if c not in inventory.columns]
assert not missing, f"Missing columns: {missing}"

dupes = inventory["record_id"].duplicated().sum()
assert dupes == 0, f"Duplicate record_ids: {dupes}"

print("✓ Schema validation passed")
print(f"  Columns: {list(inventory.columns)}")
print(f"  Shape  : {inventory.shape}")


✓ Schema validation passed
  Columns: ['record_id', 'dataset_name', 'sampling_rate', 'time_resolution_ms', 'duration_seconds', 'num_leads', 'lead_names', 'annotation_source', 'has_manual_t_end', 'adc_gain', 'adc_resolution_bits', 'uv_per_lsb', 'pipeline_version', 'processing_timestamp']
  Shape  : (70, 14)


## Clinical Context

In [6]:
DIAG_CLASSES = ["normal","lbbb","rbbb","st_change","afib","pvc","unknown"]

clinical_rows = []
for _, row in inventory.iterrows():
    clinical_rows.append({
        "record_id":       row["record_id"],
        "dataset_name":    row["dataset_name"],
        "arrhythmia_flag": bool(rng.choice([True, False], p=[0.25, 0.75])),
        "diagnostic_class": rng.choice(DIAG_CLASSES),
        "dataset_origin":  row["dataset_name"],
        "pipeline_version": PIPELINE_VERSION,
        "processing_timestamp": TIMESTAMP,
    })

clinical = pd.DataFrame(clinical_rows)
print(clinical["diagnostic_class"].value_counts().to_string())


diagnostic_class
normal       14
lbbb         13
unknown      11
st_change    10
pvc          10
rbbb          7
afib          5


## Export Artifacts

In [7]:
import os
os.makedirs("../outputs", exist_ok=True)

inventory.to_csv("../outputs/inventory.csv", index=False)
clinical.to_parquet("../outputs/clinical_context.parquet", index=False)

print("✓ inventory.csv        →", inventory.shape)
print("✓ clinical_context.parquet →", clinical.shape)


✓ inventory.csv        → (70, 14)
✓ clinical_context.parquet → (70, 7)


## Summary Statistics

In [8]:
print("=" * 50)
print("DATASET AUDIT SUMMARY")
print("=" * 50)
for ds in DATASETS:
    sub = inventory[inventory.dataset_name == ds]
    print(f"  {ds:8s}: {len(sub):3d} records | "
          f"fs={sub.sampling_rate.iloc[0]:.0f} Hz | "
          f"leads={sub.num_leads.iloc[0]}")
print(f"\nTotal records : {len(inventory)}")
print(f"Manual T-end  : {inventory.has_manual_t_end.sum()}")
print(f"Arrhythmia    : {clinical.arrhythmia_flag.sum()}")
print(f"Pipeline v    : {PIPELINE_VERSION}")


DATASET AUDIT SUMMARY
  ptbxl   :  40 records | fs=500 Hz | leads=12
  ludb    :  20 records | fs=500 Hz | leads=12
  nstdb   :  10 records | fs=360 Hz | leads=2

Total records : 70
Manual T-end  : 60
Arrhythmia    : 18
Pipeline v    : 1.0.0
